In [ ]:
# Install Tiny-SIMA and its dependencies into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not ((3, 10) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "Tiny-SIMA currently requires a Python 3.10, 3.11, or 3.12 kernel."
    )

search_roots = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in search_roots if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find pyproject.toml. Open this notebook from the cloned "
        "Tiny-SIMA repository."
    )

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode
        == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

install_target = f"{repo_root}[cuda12]" if has_cuda else str(repo_root)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "-e",
        install_target,
    ]
)

# Editable installs use a .pth file that is normally loaded at interpreter
# startup. Add src/ explicitly so this already-running kernel can import it.
source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print(f"Installed Tiny-SIMA from {repo_root}")
print(f"Added {source_root} to this kernel's import path")
print(f"CUDA dependencies requested: {has_cuda}")

# Craftax environment smoke test

This notebook checks that Craftax can reset, step, render, and run a vectorized throughput benchmark.

In [ ]:
import jax
import matplotlib.pyplot as plt

from tiny_sima.envs.craftax import benchmark_environment, make_environment, render_state

print("JAX devices:", jax.devices())

In [ ]:
bundle = make_environment(
    "Craftax-Classic-Symbolic-v1",
    max_episode_steps=256,
)
bundle.spec

In [ ]:
reset_rng, action_rng, step_rng = jax.random.split(jax.random.PRNGKey(0), 3)
observation, state = bundle.env.reset(reset_rng, bundle.params)
action = bundle.env.action_space(bundle.params).sample(action_rng)
next_observation, next_state, reward, done, info = bundle.env.step(
    step_rng,
    state,
    action,
    bundle.params,
)
{
    "observation_shape": observation.shape,
    "action": int(action),
    "reward": float(reward),
    "done": bool(done),
    "info_keys": sorted(info),
}

In [ ]:
frame = render_state(next_state, bundle.spec.name, block_pixel_size=16)
plt.figure(figsize=(8, 6))
plt.imshow(frame)
plt.axis("off")
plt.title("Craftax-Classic")
plt.show()

In [ ]:
benchmark = benchmark_environment(
    env_name=bundle.spec.name,
    num_envs=16,
    num_steps=64,
    max_episode_steps=256,
)
benchmark._asdict()